# Credit Risk Analysis

See the repository README and data/README.md before running. Outputs were cleared for publication.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(ROOT / "data")))
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Local data paths are configured in the first cell.

In [ ]:
train = pd.read_csv( DATA_DIR / "cs-training.csv")
test = pd.read_csv( DATA_DIR / "cs-test.csv")

In [ ]:
test.head()

In [ ]:
test.head()

In [ ]:
test.shape

In [ ]:
train.shape

In [ ]:
train.info()

In [ ]:
train.isnull().sum()

In [ ]:
train.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.heatmap(train.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Value Map')
plt.show()

In [ ]:
sns.countplot(x='SeriousDlqin2yrs', data=train)
plt.title('Class Distribution')
plt.show()

In [ ]:
features = train.columns.drop('SeriousDlqin2yrs')

train[features].hist(figsize=(16, 10), bins=50)
plt.tight_layout()
plt.show()

In [ ]:
## detect the Outliars
print( (train['RevolvingUtilizationOfUnsecuredLines']).describe())
print( (train['RevolvingUtilizationOfUnsecuredLines']>100).sum())
print( (train['age'] < 21 ).sum())
print( (train['age'] > 95 ).sum())


In [ ]:

print(train['NumberOfTimes90DaysLate'].value_counts().head(20))
## clean data by merge all those 3 and then seprate them again

In [ ]:
print(train['DebtRatio'].describe())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cols = ['RevolvingUtilizationOfUnsecuredLines', 'age',
        'DebtRatio', 'MonthlyIncome',
        'NumberOfTimes90DaysLate', 'NumberOfDependents']

for ax, col in zip(axes.flatten(), cols):
    sns.boxplot(y=train[col], ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# Mean default rate by binned age
train['age_bin'] = pd.cut(train['age'], bins=[0,25,35,45,55,65,120])
train.groupby('age_bin')['SeriousDlqin2yrs'].mean().plot(kind='bar')
plt.title('Default Rate by Age Group')
plt.ylabel('Default Rate')
plt.show()

In [ ]:
# Default rate vs late payment count
train.groupby('NumberOfTimes90DaysLate')['SeriousDlqin2yrs'].mean().head(20).plot(kind='bar')
plt.title('Default Rate by Times 90 Days Late')
plt.show()
train.groupby('NumberOfTime60-89DaysPastDueNotWorse')['SeriousDlqin2yrs'].mean().head(20).plot(kind='bar')
plt.title('Default Rate by Times 60-89 Days Late')
plt.show()
train.groupby('NumberOfTime30-59DaysPastDueNotWorse')['SeriousDlqin2yrs'].mean().head(20).plot(kind='bar')
plt.title('Default Rate by Times 30-59 Days Late')
plt.show()

In [ ]:
# Distribution of MonthlyIncome for defaulters vs non-defaulters
sns.kdeplot(data=train, x='MonthlyIncome', hue='SeriousDlqin2yrs', common_norm=False)
plt.xlim(0, 25000)
plt.title('Monthly Income Distribution by Target')
plt.show()
## those who have less income will tend to fail more

In [ ]:
## those who have monthly income more than 25000 are not tending to fail

In [ ]:
plt.figure(figsize=(12, 8))
corr = train.drop('age_bin', axis=1).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
print( train['DebtRatio'].describe())

In [ ]:
train = train.drop( columns = ['Unnamed: 0' ,'age_bin'] , axis = 1 )

In [ ]:
train.head()

In [ ]:
train['age'] = train['age'].replace(0, np.nan) ## 0 age is not possible

In [ ]:
## remove the 96 , 98 , from here ,
# Let the data decide the cap — use 99th percentile
debt_cap = train['DebtRatio'].quantile(0.98)
util_cap = train['RevolvingUtilizationOfUnsecuredLines'].quantile(0.98)

print(f"DebtRatio cap will be: {debt_cap}")
print(f"Utilization cap will be: {util_cap}")

train['DebtRatio'] = train['DebtRatio'].clip(upper=debt_cap)
train['RevolvingUtilizationOfUnsecuredLines'] = train['RevolvingUtilizationOfUnsecuredLines'].clip(upper=util_cap)

In [ ]:
income_cap = train['MonthlyIncome'].quantile(0.99)
train['MonthlyIncome'] = train['MonthlyIncome'].clip(upper=income_cap)
print( income_cap)


In [ ]:
past_due_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse'
]
for cols in past_due_cols:
    train[cols] = train[cols].replace({96: np.nan , 98: np.nan})

In [ ]:
train.isnull().sum()

In [ ]:
train['MonthlyIncome'].hist(bins=50)
plt.show()

In [ ]:
train['MonthlyIncome'] = train['MonthlyIncome'].fillna(train['MonthlyIncome'].median(

))

In [ ]:
print( train['NumberOfDependents'].value_counts())
print( train['NumberOfDependents'].describe())

In [ ]:
# Mode imputation — most common value is 0
train['NumberOfDependents'] = train['NumberOfDependents'].fillna(train['NumberOfDependents'].mode()[0])
train['age'] = train['age'].fillna(train['age'].median())
past_due_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse'
]

# Missing here means the sentinel 96/98 you replaced
# Most people have 0 late payments — fill with 0
for col in past_due_cols:
    train[col] = train[col].fillna(0)

In [ ]:
train.isnull().sum()## all filled

In [ ]:
## do the feature Engineering
train.head()

In [ ]:
df = train
df.head()

In [ ]:

df['TotalPastDue'] = (df['NumberOfTime30-59DaysPastDueNotWorse'] +
                      df['NumberOfTime60-89DaysPastDueNotWorse'] +
                      df['NumberOfTimes90DaysLate'])
df['WeightedPastDue'] = (df['NumberOfTime30-59DaysPastDueNotWorse'] * 1 +
                         df['NumberOfTime60-89DaysPastDueNotWorse'] * 2 +
                         df['NumberOfTimes90DaysLate'] * 3)
df['IncomePerDependent'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)
df['MonthlyDebtPayment'] = df['DebtRatio'] * df['MonthlyIncome']

# 5. Disposable income — what's left after debt payments
df['DisposableIncome'] = df['MonthlyIncome'] - df['MonthlyDebtPayment']
df['AgeBin'] = pd.cut(df['age'],
                      bins=[17, 25, 35, 45, 55, 65, 120],
                      labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])
df['IsYoungBorrower'] = (df['age'] < 30).astype(int)
df['IsHighUtilization'] = (df['RevolvingUtilizationOfUnsecuredLines'] > 0.75).astype(int)
df['HasRealEstate'] = (df['NumberRealEstateLoansOrLines'] > 0).astype(int)
df['HasDependents'] = (df['NumberOfDependents'] > 0).astype(int)
df['EverLate'] = (df['TotalPastDue'] > 0).astype(int)

In [ ]:
df['CreditToRealEstateRatio'] = df['NumberOfOpenCreditLinesAndLoans'] / (df['NumberRealEstateLoansOrLines'] + 1)
df['AgeBin'] = df['AgeBin'].astype(str)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['AgeBin'] = le.fit_transform(df['AgeBin'])

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y        # keeps same class ratio in both splits
)

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled  = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix , classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost'             : XGBClassifier(n_estimators=100, random_state=42,
                                          eval_metric='logloss', verbosity=0)
}



In [ ]:

results = {}
for name , model in models.items() :
    model.fit(X_train_scaled , y_train_bal)
    y_pred = model.predict(X_test_scaled)
    y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]

    print(f"Model: {name}")
    auc = roc_auc_score(y_test, y_pred_prob)
    print(f"AUC: {auc:.4f}")
    print( classification_report(y_test, y_pred))
    print("-" * 50)
    results[name] = {'model': model, 'auc': auc, 'y_pred': y_pred, 'y_prob': y_pred_prob}


In [ ]:
from sklearn.metrics import roc_curve, auc , f1_score
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['auc']:.4f})")

In [ ]:
# Dependencies are installed using requirements.txt.

In [ ]:
import optuna

In [ ]:
def objective( trial):
  classifier_name = trial.suggest_categorical('classifier',['RandomForest','XGBClassifier' ,'GradientBoosting'])
  if( classifier_name == 'RandomForest'):
    n_estimators = trial.suggest_int('n_estimators', 70 , 150)
    max_depth = int(trial.suggest_int('max_depth', 10, 100, log=True))
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split , random_state=42)
  elif( classifier_name == 'XGBClassifier'):
    n_estimators = trial.suggest_int('n_estimators',70,150)
    max_depth = int(trial.suggest_int('max_depth', 10, 100, log=True))

    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    model = XGBClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate , random_state=42)
  else:
    n_estimators = trial.suggest_int('n_estimators',70,150)
    max_depth = int(trial.suggest_int('max_depth', 10, 100, log=True))
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.1, log=True)
    model = GradientBoostingClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate , random_state=42)
  model.fit(X_train_scaled, y_train_bal)
  y_pred = model.predict(X_test_scaled)

  # ── Objective Metric — F1 for class 1 only ────────────────────────
  score = f1_score(y_test, y_pred, pos_label=1)
  return score

In [ ]:
study = optuna.create_study(direction='maximize')   # maximize F1
study.optimize(objective, n_trials=30)

In [ ]:
print("Best Trial:")
print(f"  F1 Score  : {study.best_value:.4f}")
print(f"  Params    : {study.best_params}")

In [ ]:
final_model = RandomForestClassifier(n_estimators=144,
                                      max_depth=10,
                                      min_samples_split=10,
                                      class_weight='balanced',
                                      random_state=42)
final_model.fit(X_train_scaled, y_train_bal)
y_pred_final = final_model.predict(X_test_scaled)
y_prob_final = final_model.predict_proba(X_test_scaled)[:, 1]


In [ ]:
import shap
import numpy as np

# ── SHAP Explainer ────────────────────────────────────────────────────
explainer   = shap.Explainer(final_model, X_train_scaled)
shap_values = explainer(X_test_scaled)


In [ ]:

# ── Plot 1: Summary Plot ──────────────────────────────────────────────
shap.summary_plot(shap_values.values[:, :, 1],
                  X_test_scaled,
                  feature_names=X.columns.tolist())

In [ ]:

# ── Plot 2: Bar Plot ──────────────────────────────────────────────────
shap.summary_plot(shap_values.values[:, :, 1],
                  X_test_scaled,
                  feature_names=X.columns.tolist(),
                  plot_type='bar')

In [ ]:

# ── Plot 3: Waterfall ─────────────────────────────────────────────────
defaulter_idx = np.where(y_test.values == 1)[0][0]  # ✅ positional index
shap.plots.waterfall(shap_values[defaulter_idx, :, 1])

In [ ]:

# ── Plot 4: Force Plot ────────────────────────────────────────────────
shap.initjs()
shap.force_plot(explainer.expected_value[1],
                shap_values.values[defaulter_idx, :, 1],
                X_test_scaled[defaulter_idx],
                feature_names=X.columns.tolist())

In [ ]:

# ── Plot 5: Dependence Plot ───────────────────────────────────────────
shap.dependence_plot('TotalPastDue',
                     shap_values.values[:,:, 1],
                     X_test_scaled,
                     feature_names=X.columns.tolist())

In [ ]:
# ── Retrain with best params ──────────────────────────────────────────
best_params = study.best_params.copy()
best_classifier = best_params.pop('classifier')

print(f"Best classifier: {best_classifier}")
print(f"Best params: {best_params}")

if best_classifier == 'RandomForest':
    final_model = RandomForestClassifier(**best_params,
                                         class_weight='balanced',
                                         random_state=42)
elif best_classifier == 'XGBClassifier':
    final_model = XGBClassifier(**best_params,
                                 eval_metric='logloss',
                                 verbosity=0,
                                 random_state=42)
else:
    final_model = GradientBoostingClassifier(**best_params,
                                              random_state=42)

final_model.fit(X_train_scaled, y_train_bal)
y_pred_final = final_model.predict(X_test_scaled)
y_prob_final = final_model.predict_proba(X_test_scaled)[:, 1]

# ── Final Evaluation ──────────────────────────────────────────────────
print("\nFinal Model Results:")
print(classification_report(y_test, y_pred_final))
print(f"AUC: {roc_auc_score(y_test, y_prob_final):.4f}")

# ── Compare Baseline vs Tuned ─────────────────────────────────────────
print("\n── Baseline vs Tuned Comparison ─────────────────────")
print(f"{'Model':25s} {'AUC':>8} {'Class1 Recall':>15} {'Class1 F1':>12}")
print("-" * 65)
for name, res in results.items():
    print(f"{name:25s} {res['auc']:>8.4f}")
print(f"{'Tuned (Optuna)':25s} {roc_auc_score(y_test, y_prob_final):>8.4f}")